<a href="https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Research question

Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring based on observable search-performance signals?

### Decision it supports

The analysis supports prioritizing pages for human review. A ranked queue can help reviewers decide which pages deserve attention first and consider an appropriate action such as refresh, expansion, protection, pruning, or monitoring.

The goal is to improve review prioritization, not to automatically decide the action or guarantee that changing a page will improve future performance.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



This analysis uses the **FlyRank internship warehouse release** available through the gated Hugging Face dataset.

### Table and grain

The main source is:

- `fact_content_daily_performance`

The table is at a **daily × client × content** grain. Each row represents the search and analytics performance of one content page for one client on one day.

### Date window

The current analysis uses **March 2026**:

- Start: `2026-03-01`
- End: `2026-03-31`

The March extract contains **9,841,378 daily rows**.

### Signals used

For the page-level analysis, the daily records are aggregated by client and content page using:

- Search impressions
- Search clicks
- Search position

These are used to calculate:

- Total impressions
- Total clicks
- CTR
- Average search position

A position-bucket CTR benchmark is also calculated from the March data.

### Exclusions

The analysis does not use:

- Client names or domains
- URLs
- Private search queries
- Credentials or private information
- Raw exports
- Future-window performance metrics
- Product flags
- The starter `trend_direction` field as a baseline input

The analysis is intended to provide **public-safe, pseudonymized, decision-support analysis** rather than identify individual clients or pages publicly.

In [1]:
import os
import duckdb
from google.colab import userdata

# Get the Hugging Face token from Colab Secrets
token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token

# Create DuckDB connection
con = duckdb.connect()

# Store the token securely inside DuckDB
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [2]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{march_path}')
""").df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [3]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
            AS unique_page_days
    FROM read_parquet('{march_path}')
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_days
0,9841378,9841378


In [4]:
schema_check = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

schema_check[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [5]:
page_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position
    FROM read_parquet('{march_path}')
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

page_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,sum_position
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,44965.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,1456.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,36794.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,36762.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,663.0


In [6]:
import numpy as np

page_df["ctr"] = page_df["clicks"] / page_df["impressions"]

# No impressions means CTR cannot be measured
page_df.loc[page_df["impressions"] == 0, "ctr"] = np.nan

page_df["avg_position"] = (
    page_df["sum_position"] / page_df["impressions"]
)

page_df.loc[page_df["impressions"] == 0, "avg_position"] = np.nan

page_df.head()

,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,avg_position
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,44965.0,0.001073,6.893301
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,1456.0,0.000000,3.214128
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,36794.0,0.001066,6.535346
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,36762.0,0.002629,7.435680
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,663.0,0.000000,15.785714


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*


### Assumptions

The analysis treats search-performance signals observed during March 2026 as evidence for prioritizing pages for human review.

A lower CTR than pages appearing in a similar search-position range is treated as a potential review signal, not proof that the page needs a specific change or that a change will improve future performance.

### Features

The page-level analysis uses observable March 2026 search-performance signals:

- Search impressions
- Search clicks
- CTR
- Impression-weighted average search position
- Position bucket
- Position-specific CTR benchmark

The position bucket is used to calculate a position-specific CTR benchmark.

### Label / target

This analysis does not use a future performance outcome as a supervised learning label for the baseline.

Instead, the baseline creates a review-priority signal from the current March 2026 observations. Therefore, the resulting score should be interpreted as a decision-support ranking rather than a prediction of future recovery.

### Baseline

The baseline identifies pages with:

1. At least 500 search impressions during March 2026.
2. CTR below the benchmark for their position bucket.

The CTR gap is calculated as:

`position-benchmark CTR − page CTR`

Pages with a positive gap are candidates for review.

The ranking score combines the CTR gap with log-transformed impressions so that visibility contributes to priority without allowing extremely large impression counts to dominate the ranking.

### Validation design

The baseline will be evaluated using ranking-oriented measures such as Precision@K where an observed target is available.

The analysis will compare the baseline with more complex models only if those models can be evaluated using the same data slice, target definition, and metric.

### Leakage checks

The analysis excludes:

- Future-window performance metrics
- Product or production action flags
- The starter `trend_direction` field
- Client names, domains, URLs, and private queries
- Any feature derived from the evaluation outcome

The purpose is to ensure that the ranking uses information that would have been observable when the review-prioritization decision was made.

In [8]:
import pandas as pd
def position_bucket(position):
    if pd.isna(position):
        return "No position"
    elif position <= 3:
        return "Top 3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    else:
        return "20+"

page_df["position_bucket"] = page_df["avg_position"].apply(position_bucket)

page_df[["avg_position", "position_bucket"]].head()

,avg_position,position_bucket
0,6.893301,4-10
1,3.214128,4-10
2,6.535346,4-10
3,7.435680,4-10
4,15.785714,11-20


In [9]:
position_benchmark = (
    page_df.groupby("position_bucket")
    .agg(
        benchmark_ctr=("clicks", "sum")
    )
)

position_benchmark["total_impressions"] = (
    page_df.groupby("position_bucket")["impressions"].sum()
)

position_benchmark["benchmark_ctr"] = (
    position_benchmark["benchmark_ctr"]
    / position_benchmark["total_impressions"]
)

position_benchmark

,benchmark_ctr,total_impressions
position_bucket,,
11-20,0.003158,31191659.0
20+,0.001361,59933354.0
4-10,0.003249,148112436.0
No position,NaN,0.0
Top 3,0.003876,41420140.0


In [10]:
page_df["position_benchmark_ctr"] = (
    page_df["position_bucket"]
    .map(position_benchmark["benchmark_ctr"])
)

page_df["ctr_gap"] = (
    page_df["position_benchmark_ctr"] - page_df["ctr"]
)

page_df[
    [
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap"
    ]
].head()

,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap
0,content_7a105f548d9c6916,6523.0,0.001073,6.893301,4-10,0.003249,0.002176
1,content_a3ea9792f793ec72,453.0,0.000000,3.214128,4-10,0.003249,0.003249
2,content_36c36abc7650d7af,5630.0,0.001066,6.535346,4-10,0.003249,0.002183
3,content_a7da352b73b02668,4944.0,0.002629,7.435680,4-10,0.003249,0.000619
4,content_f39be42b42a4e8f6,42.0,0.000000,15.785714,11-20,0.003158,0.003158


In [11]:
ranked_df = page_df[
    (page_df["impressions"] >= 500) &
    (page_df["ctr_gap"] > 0)
].copy()

print("Candidate pages:", len(ranked_df))

ranked_df[
    [
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap"
    ]
].head()

Candidate pages: 41087


,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap
0,content_7a105f548d9c6916,6523.0,0.001073,6.893301,4-10,0.003249,0.002176
2,content_36c36abc7650d7af,5630.0,0.001066,6.535346,4-10,0.003249,0.002183
3,content_a7da352b73b02668,4944.0,0.002629,7.435680,4-10,0.003249,0.000619
9,content_aafb2ab7e5fc80d0,7709.0,0.002594,5.127643,4-10,0.003249,0.000654
10,content_20403327d8d9374c,3561.0,0.002808,9.273799,4-10,0.003249,0.000441


In [12]:
ranked_df["score"] = (
    np.log1p(ranked_df["impressions"]) *
    ranked_df["ctr_gap"]
)

ranked_df = ranked_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap",
        "score"
    ]
].head(20)

,rank,content_hash_id,impressions,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap,score
0,1,content_44f34c0a90047651,212404.0,0.000113,0.665877,Top 3,0.003876,0.003763,0.046163
1,2,content_8e1334d6356668e3,134984.0,0.000007,2.693038,Top 3,0.003876,0.003869,0.045704
2,3,content_fec55986a1868d62,124075.0,0.000008,0.308426,Top 3,0.003876,0.003868,0.045371
3,4,content_9c057b66c30a3abb,83834.0,0.000012,0.116003,Top 3,0.003876,0.003864,0.043810
4,5,content_bf078007df823490,44707.0,0.000000,1.400049,Top 3,0.003876,0.003876,0.041508
5,6,content_d61fc394d10cba41,38000.0,0.000026,2.362579,Top 3,0.003876,0.003850,0.040601
6,7,content_fc67675904376267,60172.0,0.000299,2.126022,Top 3,0.003876,0.003577,0.039368
7,8,content_dc91779c3d085398,25625.0,0.000039,2.389151,Top 3,0.003876,0.003837,0.038955
8,9,content_306bc78dff1eb683,80821.0,0.000433,1.444266,Top 3,0.003876,0.003443,0.038910
9,10,content_66bf45eb0c5bb550,24259.0,0.000041,2.071726,Top 3,0.003876,0.003835,0.038722


In [16]:
ranked_df["reason_code"] = "HIGH_VISIBILITY_LOW_CTR"

ranked_df["action"] = "REVIEW_CTR_OPPORTUNITY"

ranked_df["why_its_here"] = (
    "CTR is below the benchmark for the page's position range "
    "with at least 500 impressions."
)

ranked_df["what_would_make_it_wrong"] = (
    "The CTR gap may reflect query mix, page intent, or other factors "
    "not captured by this baseline."
)

ranked_df["confidence_note"] = (
    "Decision-support signal; not a prediction of future performance."
)

ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
].head(20)



,rank,content_hash_id,impressions,ctr,avg_position,ctr_gap,score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,0.000113,0.665877,0.003763,0.046163,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
1,2,content_8e1334d6356668e3,134984.0,0.000007,2.693038,0.003869,0.045704,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
2,3,content_fec55986a1868d62,124075.0,0.000008,0.308426,0.003868,0.045371,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
3,4,content_9c057b66c30a3abb,83834.0,0.000012,0.116003,0.003864,0.043810,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
4,5,content_bf078007df823490,44707.0,0.000000,1.400049,0.003876,0.041508,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
5,6,content_d61fc394d10cba41,38000.0,0.000026,2.362579,0.003850,0.040601,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
6,7,content_fc67675904376267,60172.0,0.000299,2.126022,0.003577,0.039368,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
7,8,content_dc91779c3d085398,25625.0,0.000039,2.389151,0.003837,0.038955,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
8,9,content_306bc78dff1eb683,80821.0,0.000433,1.444266,0.003443,0.038910,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
9,10,content_66bf45eb0c5bb550,24259.0,0.000041,2.071726,0.003835,0.038722,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY


In [18]:
print(ranked_df.loc[0, "why_its_here"])
print(ranked_df.loc[0, "what_would_make_it_wrong"])
print(ranked_df.loc[0, "confidence_note"])

CTR is below the benchmark for the page's position range with at least 500 impressions.
The CTR gap may reflect query mix, page intent, or other factors not captured by this baseline.
Decision-support signal; not a prediction of future performance.


In [19]:
baseline_top20 = ranked_df[
    [
        "rank",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_bucket",
        "position_benchmark_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
].head(20)

baseline_top20

,rank,content_hash_id,impressions,clicks,ctr,avg_position,position_bucket,position_benchmark_ctr,ctr_gap,score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,24.0,0.000113,0.665877,Top 3,0.003876,0.003763,0.046163,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
1,2,content_8e1334d6356668e3,134984.0,1.0,0.000007,2.693038,Top 3,0.003876,0.003869,0.045704,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
2,3,content_fec55986a1868d62,124075.0,1.0,0.000008,0.308426,Top 3,0.003876,0.003868,0.045371,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
3,4,content_9c057b66c30a3abb,83834.0,1.0,0.000012,0.116003,Top 3,0.003876,0.003864,0.043810,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
4,5,content_bf078007df823490,44707.0,0.0,0.000000,1.400049,Top 3,0.003876,0.003876,0.041508,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
5,6,content_d61fc394d10cba41,38000.0,1.0,0.000026,2.362579,Top 3,0.003876,0.003850,0.040601,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
6,7,content_fc67675904376267,60172.0,18.0,0.000299,2.126022,Top 3,0.003876,0.003577,0.039368,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
7,8,content_dc91779c3d085398,25625.0,1.0,0.000039,2.389151,Top 3,0.003876,0.003837,0.038955,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
8,9,content_306bc78dff1eb683,80821.0,35.0,0.000433,1.444266,Top 3,0.003876,0.003443,0.038910,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
9,10,content_66bf45eb0c5bb550,24259.0,1.0,0.000041,2.071726,Top 3,0.003876,0.003835,0.038722,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY


In [20]:
baseline_summary = (
    ranked_df.head(20)
    .groupby(["reason_code", "action"])
    .size()
    .reset_index(name="pages")
)

baseline_summary

,reason_code,action,pages
0,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY,20


In [21]:
baseline_columns = ranked_df.columns.tolist()

leakage_columns = [
    "trend_direction",
    "future_ctr",
    "future_impressions",
    "future_clicks",
    "product_flag"
]

found_leakage = [
    col for col in leakage_columns
    if col in baseline_columns
]

print("Potential leakage columns found:", found_leakage)

Potential leakage columns found: []


In [22]:
top20_stats = ranked_df.head(20)[
    ["impressions", "ctr", "avg_position", "ctr_gap", "score"]
].describe()

top20_stats

,impressions,ctr,avg_position,ctr_gap,score
count,20.000000,20.000000,20.000000,20.000000,20.000000
mean,57846.650000,0.000134,2.071451,0.003712,0.039384
std,51634.940548,0.000172,1.592242,0.000208,0.003386
min,10886.000000,0.000000,0.116003,0.003204,0.036033
25%,24044.750000,0.000008,1.136199,0.003659,0.036606
50%,34787.500000,0.000043,2.080271,0.003799,0.038473
75%,81574.250000,0.000185,2.414477,0.003869,0.040828
max,212404.000000,0.000597,7.831807,0.003876,0.046163


In [23]:
evaluation_columns = [
    "content_hash_id",
    "rank",
    "score",
    "reason_code",
    "action"
]

evaluation_df = ranked_df[evaluation_columns].copy()

evaluation_df.head(20)

,content_hash_id,rank,score,reason_code,action
0,content_44f34c0a90047651,1,0.046163,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
1,content_8e1334d6356668e3,2,0.045704,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
2,content_fec55986a1868d62,3,0.045371,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
3,content_9c057b66c30a3abb,4,0.043810,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
4,content_bf078007df823490,5,0.041508,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
5,content_d61fc394d10cba41,6,0.040601,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
6,content_fc67675904376267,7,0.039368,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
7,content_dc91779c3d085398,8,0.038955,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
8,content_306bc78dff1eb683,9,0.038910,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY
9,content_66bf45eb0c5bb550,10,0.038722,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR_OPPORTUNITY


In [24]:
# Check whether an observed target is available in our current page-level data

target_candidates = [
    col for col in page_df.columns
    if any(term in col.lower() for term in [
        "trend",
        "future",
        "target",
        "label",
        "outcome"
    ])
]

print("Possible target columns:", target_candidates)

Possible target columns: []


### Evaluation constraint

The March 2026 warehouse slice does not contain an observed future-outcome or target column that can be used to calculate Precision@K for the baseline.

Therefore, Precision@K is not reported for this current-window baseline.

The baseline is treated as a transparent decision-support ranking based on observed March 2026 search-performance signals. Any future evaluation would require a separately defined and temporally aligned outcome, such as a subsequent performance window or another independently observed review outcome.

In [28]:
files = con.sql("""
    SELECT file
    FROM glob(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/*.parquet'
    )
    ORDER BY file
""").df()

files

,file
0,hf://datasets/FlyRank/internship-warehouse/fac...
1,hf://datasets/FlyRank/internship-warehouse/fac...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [29]:
files["month"] = files["file"].str.extract(r"month=(\d{4}-\d{2})")

files[["month"]].drop_duplicates().sort_values("month")

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


In [30]:
april_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

march_pages = con.sql(f"""
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM read_parquet('{march_path}')
""")

april_pages = con.sql(f"""
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM read_parquet('{april_path}')
""")

overlap = con.sql("""
    SELECT COUNT(*) AS overlapping_pages
    FROM march_pages m
    INNER JOIN april_pages a
        USING (client_hash_id, content_hash_id)
""").df()

overlap

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_pages
0,331436


In [31]:
march_april_sample = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks
        FROM read_parquet('{march_path}')
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{april_path}')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.*,
        a.april_impressions,
        a.april_clicks
    FROM march m
    INNER JOIN april a
        USING (client_hash_id, content_hash_id)
    LIMIT 10
""").df()

march_april_sample

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,march_clicks,april_impressions,april_clicks
0,client_62f4a7e64f5e0096,content_35d979572550dd7f,1423.0,1.0,1180.0,0.0
1,client_62f4a7e64f5e0096,content_56a4dabd555eec90,1.0,0.0,15.0,0.0
2,client_62f4a7e64f5e0096,content_e174e46a5b0733a8,0.0,0.0,0.0,0.0
3,client_62f4a7e64f5e0096,content_dc2422b7fc475fd6,0.0,0.0,0.0,0.0
4,client_62f4a7e64f5e0096,content_89edcdb8887fe6db,0.0,0.0,0.0,0.0
5,client_62f4a7e64f5e0096,content_d95e1739253d6a65,208.0,0.0,168.0,0.0
6,client_62f4a7e64f5e0096,content_405cc2e0475690e9,607.0,0.0,249.0,0.0
7,client_62f4a7e64f5e0096,content_fdb0d09a710f671c,267.0,0.0,336.0,0.0
8,client_62f4a7e64f5e0096,content_a7b1f7921f001dce,1489.0,0.0,832.0,0.0
9,client_62f4a7e64f5e0096,content_f941aaa482eeae55,0.0,0.0,0.0,0.0


In [32]:
march_april_stats = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS pages,
    SUM(CASE WHEN march_impressions > 0 THEN 1 ELSE 0 END) AS march_visible,
    SUM(CASE WHEN april_impressions > 0 THEN 1 ELSE 0 END) AS april_visible,
    SUM(CASE
        WHEN march_impressions > 0
         AND april_impressions > 0
        THEN 1 ELSE 0
    END) AS visible_both_months
FROM march
INNER JOIN april
USING (client_hash_id, content_hash_id)
""").df()

march_april_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pages,march_visible,april_visible,visible_both_months
0,331436,176737.0,176441.0,158549.0


In [33]:
change_stats = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS pages,
    AVG(april_impressions - march_impressions) AS avg_impression_change,
    AVG(
        CASE
            WHEN march_impressions > 0
            THEN (april_impressions - march_impressions) / march_impressions
        END
    ) AS avg_impression_pct_change,
    SUM(
        CASE
            WHEN april_impressions < march_impressions
            THEN 1 ELSE 0
        END
    ) AS pages_with_impression_decline
FROM march
INNER JOIN april
USING (client_hash_id, content_hash_id)
WHERE march_impressions > 0
  AND april_impressions > 0
""").df()

change_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pages,avg_impression_change,avg_impression_pct_change,pages_with_impression_decline
0,158549,9.714524,1.954214,93779.0


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
